In [ ]:
import  numpy

Load Data

In [ ]:
data_x_spectrum = numpy.load("x.npz")

In [ ]:
frequency = data_x_spectrum['frequency']
amplitude = data_x_spectrum['amplitude']

In [ ]:
#cleaning up the data
import numpy as np


def remove_edge_artifacts(
    frequency,
    amplitude,
    mean_window=501,
    gradient_window=201,
    gradient_threshold_factor=0.1,
    amplitude_threshold_factor=5,
    min_region=200,
    symmetric_edges=True,
    debug=True
):

    frequency = np.asarray(frequency)
    amplitude = np.asarray(amplitude)


    if mean_window % 2 == 0:
        mean_window += 1


    half = mean_window // 2


    # derivative

    grad = np.zeros_like(amplitude)


    for i in range(
        half,
        len(amplitude)-half
    ):

        left_mean = np.mean(
            amplitude[i-half:i]
        )

        right_mean = np.mean(
            amplitude[i:i+half]
        )

        grad[i] = right_mean - left_mean



    # mean value of the derivative

    kernel = np.ones(
        gradient_window
    ) / gradient_window


    grad_mean = np.convolve(
        grad,
        kernel,
        mode="same"
    )


    abs_grad = np.abs(grad_mean)


    gradient_threshold = (
        gradient_threshold_factor
        *
        np.max(abs_grad)
    )


    background = np.median(
        amplitude[
            len(amplitude)//4:
            3*len(amplitude)//4
        ]
    )


    amplitude_threshold = (
        amplitude_threshold_factor
        *
        background
    )


    if debug:

        print("max gradient:",
              np.max(abs_grad))

        print("gradient threshold:",
              gradient_threshold)

        print("background:",
              background)

        print("amplitude threshold:",
              amplitude_threshold)



    left = 0


    for i in range(
        0,
        len(amplitude)-min_region
    ):

        grad_region = np.mean(
            abs_grad[i:i+min_region]
        )

        amp_region = np.mean(
            amplitude[i:i+min_region]
        )


        if (
            grad_region < gradient_threshold
            and
            amp_region < amplitude_threshold
        ):

            left = i
            break



    right = len(amplitude)-1


    for i in range(
        len(amplitude)-1,
        min_region,
        -1
    ):

        grad_region = np.mean(
            abs_grad[i-min_region:i]
        )

        amp_region = np.mean(
            amplitude[i-min_region:i]
        )


        if (
            grad_region < gradient_threshold
            and
            amp_region < amplitude_threshold
        ):

            right = i
            break



    # local peak safe, so peaks on the edge don't get cut off

    def has_local_peak(
        region,
        drop_fraction=0.5
    ):

        if len(region) < 50:
            return False


        peak_idx = np.argmax(region)

        peak_val = region[peak_idx]


        if peak_val <= 0:
            return False


        left_side = region[:peak_idx]

        right_side = region[peak_idx:]


        if (
            len(left_side) < 10
            or
            len(right_side) < 10
        ):
            return False


        left_min = np.min(
            left_side
        )

        right_min = np.min(
            right_side
        )


        return (
            left_min < drop_fraction * peak_val
            and
            right_min < drop_fraction * peak_val
        )



    edge_length = min_region * 5

    # right edge

    right_region = amplitude[
        max(0, right-edge_length):
        right
    ]


    if has_local_peak(right_region):

        if debug:
            print(
                "Right side contains local peak -> keep peak"
            )

        peak_idx = np.argmax(
            right_region
        )

        right = (
            max(0, right-edge_length)
            +
            peak_idx
        )



    # left edge

    left_region = amplitude[
        left:
        min(
            len(amplitude),
            left+edge_length
        )
    ]


    if has_local_peak(left_region):

        if debug:
            print(
                "Left side contains local peak -> keep peak"
            )

        peak_idx = np.argmax(
            left_region
        )

        left = (
            left
            +
            peak_idx
        )

    # symmetric cut

    if symmetric_edges:

        left_cut = left

        right_cut = (
            len(amplitude)
            -
            right
            -
            1
        )


        symmetric_cut = min(
            left_cut,
            right_cut
        )


        if debug:

            print("="*60)
            print("Symmetric edge correction")
            print("="*60)

            print(
                f"Detected cuts: "
                f"left={left_cut}, "
                f"right={right_cut}"
            )

            print(
                f"Applied symmetric cut: "
                f"{symmetric_cut}"
            )


        left = symmetric_cut

        right = (
            len(amplitude)
            -
            symmetric_cut
            -
            1
        )


    if debug:

        print("="*60)
        print("Edge artifact removal")
        print("="*60)

        print(
            f"Left cut : {left}"
        )

        print(
            f"Right cut: {len(amplitude)-right-1}"
        )

        print(
            f"Remaining points: {right-left+1}"
        )


    return (
        frequency[left:right+1],
        amplitude[left:right+1]
    )

In [ ]:
frequency, amplitude = remove_edge_artifacts(
    frequency,
    amplitude,
    mean_window=1001,
    gradient_window=501,
    gradient_threshold_factor=0.15,
    amplitude_threshold_factor=20,
    min_region=500,
    symmetric_edges=True,
    debug=True
)

In [ ]:
import numpy as np

# Automatic parameter estimation

def estimate_spectrum_parameters(
    frequency,
    expected_frev=2.00e6,  # values for ESR- change if needed
    frev_tolerance=0.10e6,  # values for ESR
    safety_margin=2
):

    f_min = np.min(frequency)
    f_max = np.max(frequency)

    bandwidth = f_max - f_min

    # number of possible repetitions 

    min_repeats = max(
        3,
        int(np.floor(bandwidth / (expected_frev + frev_tolerance)))
    )
    min_repeats = min_repeats-1
    max_repeats = int(
        np.ceil(
            bandwidth /
            (expected_frev - frev_tolerance)
        )
    ) + safety_margin

    # expected harmoincs

    harmonic_min = int(
        np.floor(
            (f_min /
            (expected_frev + frev_tolerance))-5
        )
    )

    harmonic_max = int(
        np.ceil(
            (f_max /
            (expected_frev - frev_tolerance))+5
        )
    )


    print("="*60)
    print("Automatic spectrum parameters")
    print("="*60)

    print(f"Spectrum range : {f_min/1e6:.3f} - {f_max/1e6:.3f} MHz")
    print(f"Bandwidth      : {bandwidth/1e6:.3f} MHz")
    print()
    print(f"Expected f_rev : {expected_frev/1e6:.3f} MHz")
    print(f"Tolerance      : ±{frev_tolerance/1e6:.3f} MHz")
    print()
    print(f"Harmonics      : {harmonic_min} ... {harmonic_max}")
    print(f"Repeats        : {min_repeats} ... {max_repeats}")

    return {
        "freq_min": expected_frev-frev_tolerance,
        "freq_max": expected_frev+frev_tolerance,
        "harmonic_min": harmonic_min,
        "harmonic_max": harmonic_max,
        "min_repeats": min_repeats,
        "max_repeats": max_repeats
    }

In [ ]:
params = estimate_spectrum_parameters(
    frequency,
    expected_frev=2.00e6,
    frev_tolerance=0.10e6
)

In [ ]:
# optional : Plot of the spectrum 

# import plotly.graph_objects
# fig = plotly.graph_objects.Figure()
# fig.add_trace(plotly.graph_objects.Scatter(x=frequency,
#     y=amplitude,
#     mode="lines",
#     name="Spectrum"
# ))
# fig.update_layout(
#     title="Interactive Spectrum",
#     xaxis_title="frequency",
#     yaxis_title="amplitude"
# )


# fig.show()


Peakfinder

In [ ]:
import numpy as np
from scipy.signal import find_peaks


def find_peaks_adaptive(
    frequency,
    amplitude,
    prominence_resonance=0.003,
    distance_resonance=10,
    threshold_background=0.001,
    merge_background_points=7,
    activity_window=300,
    activity_fraction=0.40
):
    # rough estimation of peaks with find_peaks

    rough_peaks, props = find_peaks(
        amplitude,
        prominence=max(
            prominence_resonance * 0.3,
            threshold_background
        ),
        distance=max(
            2,
            distance_resonance // 2
        )
    )


    activity = np.zeros(len(amplitude))

    half = activity_window // 2


    for peak, prom in zip(
        rough_peaks,
        props["prominences"]
    ):

        left = max(
            0,
            peak-half
        )

        right = min(
            len(amplitude),
            peak+half
        )

        activity[left:right] += 1
        activity[left:right] += prom



    kernel = np.ones(activity_window) / activity_window

    activity = np.convolve(
        activity,
        kernel,
        mode="same"
    )


    # resonance area

    center = np.argmax(activity)

    limit = activity_fraction * activity[center]


    left = center
    while left > 0 and activity[left] > limit:
        left -= 1


    right = center
    while right < len(activity)-1 and activity[right] > limit:
        right += 1


    resonance_mask = np.zeros(
        len(amplitude),
        dtype=bool
    )

    resonance_mask[left:right+1] = True



    # peaks in the resonance area 

    resonance_indices = np.where(
        resonance_mask
    )[0]


    peaks_res_local, _ = find_peaks(
        amplitude[resonance_mask],
        prominence=prominence_resonance,
        distance=distance_resonance
    )


    peaks_res = resonance_indices[
        peaks_res_local
    ]



    # peaks in the "background" (not resonance area)

    background_indices = np.where(
        ~resonance_mask
    )[0]


    peaks_background = []


    for i in background_indices:

        if i == 0 or i == len(amplitude)-1:
            continue


        if (
            amplitude[i-1]
            <
            amplitude[i]
            >
            amplitude[i+1]
        ):

            if amplitude[i] > threshold_background:

                peaks_background.append(i)



    peaks_background = np.asarray(
        peaks_background,
        dtype=int
    )



    # merging 

    def merge_peaks_fast(
        peaks,
        amplitude,
        merge_points
    ):

        if len(peaks) == 0:
            return peaks


        peaks = np.sort(peaks)

        merged = []

        group = [
            peaks[0]
        ]


        for p in peaks[1:]:

            if p - group[-1] <= merge_points:

                group.append(p)

            else:

                group = np.asarray(group)

                merged.append(
                    group[
                        np.argmax(
                            amplitude[group]
                        )
                    ]
                )

                group = [p]


        group = np.asarray(group)

        merged.append(
            group[
                np.argmax(
                    amplitude[group]
                )
            ]
        )


        return np.asarray(
            merged,
            dtype=int
        )



    peaks_background = merge_peaks_fast(
        peaks_background,
        amplitude,
        merge_background_points
    )



    # combined Peaks

    peaks_all = np.unique(
        np.concatenate(
            [
                peaks_res,
                peaks_background
            ]
        )
    )


    return (
        peaks_all,
        resonance_mask,
        activity
    )



def get_adaptive_peaks(
    frequency,
    amplitude
):

    peaks, resonance_mask, activity = find_peaks_adaptive(
        frequency,
        amplitude,
        prominence_resonance=0.003,
        distance_resonance=10,
        threshold_background=0.001,
        merge_background_points=7,
        activity_window=300,
        activity_fraction=0.40
    )


    return (
        np.asarray(peaks, dtype=int),
        resonance_mask,
        activity
    )


In [ ]:
peaks, resonance_mask, activity = find_peaks_adaptive(
        frequency,
        amplitude,
        prominence_resonance=0.003,   # these need to be adapted on the data (look in the spectrum an change to get all necesary peaks)
        distance_resonance=10,
        threshold_background=0.001,
        merge_background_points=7,
        activity_window=300,
        activity_fraction=0.40
    )

In [ ]:
import plotly.graph_objects as go

def plot_spectrum_with_peaks_plotly(
    frequency,
    amplitude,
    peaks,
    f_min=None,
    f_max=None
):

    fig = go.Figure()

    # Spectrum
    fig.add_trace(go.Scatter(
        x=frequency,
        y=amplitude,
        mode="lines",
        name="Spektrum"
    ))

    fig.add_trace(go.Scatter(
        x=frequency[peaks],
        y=amplitude[peaks],
        mode="markers",
        marker=dict(color="red", size=6),
        name="Peaks"
    ))

    if f_min is not None and f_max is not None:
        fig.update_xaxes(range=[f_min, f_max])

    fig.update_layout(
        title="Spektrum mit Peaks",
        xaxis_title="Frequency",
        yaxis_title="Amplitude"
    )

    fig.show()

In [ ]:
# Plot in oder to see if the values are set correctly 
plot_spectrum_with_peaks_plotly(
    frequency,
    amplitude,
    peaks
)

Clustering of all peaks 
-> in order to compare structures and not only single peaks 

In [ ]:
def build_peak_space(
    frequency,
    amplitude,
    peaks,
    activity
):

    peak_freqs = np.asarray(
        frequency[peaks],
        dtype=float
    )

    peak_amps = np.asarray(
        amplitude[peaks],
        dtype=float
    )

    peak_activity = np.asarray(
        activity[peaks],
        dtype=float
    )


    order = np.argsort(
        peak_freqs
    )


    return (
        peaks[order],
        peak_freqs[order],
        peak_amps[order],
        peak_activity[order]
    )

In [ ]:
def cluster_peaks_by_frequency(
    peaks,
    frequency,
    amplitude,
    peak_to_group,
    cluster_gap_threshold=1.0e5,
    amp_ratio_threshold=0.6,
    valley_threshold=0.4,
    group_weight=0.5
):

    if len(peaks) == 0:
        return []


    peaks = np.asarray(
        peaks,
        dtype=int
    )


    peaks = peaks[
        np.argsort(
            frequency[peaks]
        )
    ]


    clusters = []

    current = [
        peaks[0]
    ]



    for i in range(1, len(peaks)):


        p_prev = current[-1]
        p_curr = peaks[i]


        gap = (
            frequency[p_curr]
            -
            frequency[p_prev]
        )

        # Group Information

        g_prev = peak_to_group.get(
            p_prev,
            None
        )

        g_curr = peak_to_group.get(
            p_curr,
            None
        )


        same_group = (
            g_prev is not None
            and
            g_prev == g_curr
        )

        # Amplitude
        
        amp_ratio = (
            amplitude[p_curr]
            /
            (amplitude[p_prev]+1e-12)
        )


        similar_height = (

            amp_ratio > amp_ratio_threshold

            and

            amp_ratio < 1/amp_ratio_threshold

        )
        # Valley

        window = amplitude[
            p_prev:p_curr+1
        ]


        valley_min = np.min(
            window
        )


        peak_min = min(
            amplitude[p_prev],
            amplitude[p_curr]
        )


        valley_ratio = (

            valley_min
            /
            (peak_min+1e-12)

        )


        deep_valley = (
            valley_ratio < valley_threshold
        )



        # ----------------------------------
        # Score
        # ----------------------------------

        score = 0


        if gap <= cluster_gap_threshold:
            score += 1


        if same_group:
            score += group_weight

        else:
            score -= group_weight



        if similar_height:
            score += 0.5


        if not deep_valley:
            score += 0.5



        if score >= 1.0:

            current.append(
                p_curr
            )

        else:

            clusters.append(
                current
            )

            current = [
                p_curr
            ]



    clusters.append(
        current
    )


    print(
        "Clusters:",
        len(clusters)
    )

    print(
        "Cluster sizes:",
        [
            len(c)
            for c in clusters
        ]
    )


    return clusters

In [ ]:
def run_pipeline_no_grouping(
    frequency,
    amplitude
):

    # peak detection

    peaks, resonance_mask, activity = get_adaptive_peaks(
        frequency,
        amplitude
    )

    # peak space 

    peak_freqs = frequency[peaks]

    peak_amps = amplitude[peaks]

    peak_activity = activity[peaks]


    # Sorting 

    order = np.argsort(
        peak_freqs
    )

    peaks = peaks[order]

    peak_freqs = peak_freqs[order]

    peak_amps = peak_amps[order]

    peak_activity = peak_activity[order]


    peak_to_group = {}



    clusters = cluster_peaks_by_frequency(
        peaks=np.arange(len(peaks)),
        frequency=peak_freqs,
        amplitude=peak_amps,
        peak_to_group=peak_to_group,
        cluster_gap_threshold=1.0e5,
        amp_ratio_threshold=0.6,
        valley_threshold=0.4
    )


    # Originalindices

    clusters_original = [
        peaks[c]
        for c in clusters
    ]


    print()
    print("----------------------------")
    print("No grouping pipeline")
    print("----------------------------")

    print(
        "Detected peaks:",
        len(peaks)
    )

    print(
        "Detected clusters:",
        len(clusters_original)
    )


    print(
        "Peaks in clusters:",
        sum(
            len(c)
            for c in clusters_original
        )
    )


    return (
        peaks,
        peak_freqs,
        peak_amps,
        peak_activity,
        clusters_original,
        resonance_mask,
        activity
    )

In [ ]:
peaks, resonance_mask, activity = find_peaks_adaptive(
        frequency,
        amplitude,
        prominence_resonance=0.003,
        distance_resonance=10,
        threshold_background=0.001,
        merge_background_points=7,
        activity_window=300,
        activity_fraction=0.40
    )


print("Detected peaks:", len(peaks))


(
    peaks,
    peak_freqs,
    peak_amps,
    peak_activity
) = build_peak_space(
    frequency,
    amplitude,
    peaks,
    activity
)


peak_to_group = {}



clusters = cluster_peaks_by_frequency(
    peaks=np.arange(len(peak_freqs)),
    frequency=peak_freqs,
    amplitude=peak_amps,
    peak_to_group=peak_to_group,
    cluster_gap_threshold=0.5e5,
    amp_ratio_threshold=0.6,
    valley_threshold=0.4
)



# zurück zu Originalindizes
clusters_original = [
    peaks[c]
    for c in clusters
]


print()
print("================================")
print("Cluster result")
print("================================")
print("Number of clusters:", len(clusters_original))
print(
    "Cluster sizes:",
    [len(c) for c in clusters_original]
)

In [ ]:
print("Number of Peaks:", len(peaks))
print("Number of Clusters:", len(clusters))

print(
    "Peaks in Clusters:",
    sum(len(c) for c in clusters)
)

In [ ]:
# left Peak in the cluster and right peak in the cluster for the window 
# width and flank factor and if the window collapses a fallback of 5 indices
def extract_cluster_windows(frequency, clusters, flank_factor=0.2):

    windows = []
    n = len(frequency)

    for cluster in clusters:

        if len(cluster) == 0:
            continue

        # single peaks are also allowed!
        left_idx = cluster[0]
        right_idx = cluster[-1]

        f_left = frequency[left_idx]
        f_right = frequency[right_idx]

        width = f_right - f_left

        if width == 0:
            width = (frequency[1] - frequency[0]) * 10  

        f_start = f_left - flank_factor * width
        f_end   = f_right + flank_factor * width

        start_idx = np.searchsorted(frequency, f_start)
        end_idx   = np.searchsorted(frequency, f_end)

        # Clipping
        start_idx = max(0, start_idx)
        end_idx = min(n - 1, end_idx)

    
        if end_idx <= start_idx:
            center = left_idx
            start_idx = max(0, center - 5)
            end_idx = min(n - 1, center + 5)

        windows.append((start_idx, end_idx))

    return windows


In [ ]:
# using the derivative to refine the windows -> end if derivative gets small
# patience how many points under the threshold are okay
# max_width: how wide the window can be max
def refine_window_with_derivative(
    amplitude,
    peak_idx,
    gradient,
    max_width=80,
    rel_slope=0.3,
    patience=3
):

    grad = gradient
    peak_grad = grad[peak_idx]

    threshold = rel_slope * peak_grad

    left = peak_idx
    right = peak_idx

    # links
    below_count = 0
    while left > 1:
        if grad[left] < threshold:
            below_count += 1
            if below_count >= patience:
                break
        else:
            below_count = 0

        if peak_idx - left > max_width:
            break

        left -= 1

    # rechts
    below_count = 0
    while right < len(amplitude) - 2:
        if grad[right] < threshold:
            below_count += 1
            if below_count >= patience:
                break
        else:
            below_count = 0

        if right - peak_idx > max_width:
            break

        right += 1

    return left, right

In [ ]:
# frequency width half maximum
# not enough points, return 0 , linear interpolation 
def compute_fwhm(x, y):

    x = np.asarray(x)
    y = np.asarray(y)

    half_max = np.max(y) / 2

    above = np.where(y >= half_max)[0]

    if len(above) < 2:
        return 0.0

    i_left = above[0]
    i_right = above[-1]

    if i_left == 0:
        x_left = x[0]
    else:
        x1, x2 = x[i_left - 1], x[i_left]
        y1, y2 = y[i_left - 1], y[i_left]

        x_left = x1 + (half_max - y1) * (x2 - x1) / (y2 - y1)

    if i_right == len(y) - 1:
        x_right = x[-1]
    else:
        x1, x2 = x[i_right], x[i_right + 1]
        y1, y2 = y[i_right], y[i_right + 1]

        x_right = x1 + (half_max - y1) * (x2 - x1) / (y2 - y1)

    return x_right - x_left

In [ ]:
# position error for the peaks with fwhm
def estimate_position_error(fwhm):

    return fwhm / (2 * np.sqrt(2 * np.log(2)))

In [ ]:
from scipy.signal import find_peaks
# single peak or double peak? 
# merging if more than two peaks and weighted mean value for the position / the peak
def analyze_peak_shape_raw(x, y):

    peaks, _ = find_peaks(
        y,
        prominence=np.max(y) * 0.1
    )

    if len(peaks) == 1:

        idx = peaks[0]

        peak_pos = x[idx]
        peak_amp = y[idx]

        fwhm = compute_fwhm(x, y)
        pos_err = estimate_position_error(fwhm)

        return {
            "type": "single",
            "peak_position": peak_pos,
            "amplitude": peak_amp,
            "fwhm": fwhm,
            "position_error": pos_err
        }

    elif len(peaks) >= 2:

        peak_heights = y[peaks]

        top2_idx = np.argsort(peak_heights)[-2:]

        p1, p2 = peaks[top2_idx]

        x1, x2 = x[p1], x[p2]
        y1, y2 = y[p1], y[p2]

        pos = (x1*y1 + x2*y2) / (y1 + y2)

        fwhm = compute_fwhm(x, y)

        pos_err = estimate_position_error(fwhm)

        return {
            "type": "double",
            "peak_position": pos,
            "peaks_positions": (x1, x2),
            "amplitudes": (y1, y2),
            "fwhm": fwhm,
            "position_error": pos_err
        }

    return None

In [ ]:
def extract_peak_segments_raw(
    frequency,
    amplitude,
    clusters,
    peak_to_group
):

    gradient = np.abs(np.gradient(amplitude))

    coarse_windows = extract_cluster_windows(
        frequency,
        clusters
    )

    results = []

    for i, cluster in enumerate(clusters):

        if len(cluster) == 0:
            continue

        if i >= len(coarse_windows):
            continue

        left_c, right_c = coarse_windows[i]

        # Center = aroung tallest peak

        peak_center = cluster[
            np.argmax(
                amplitude[cluster]
            )
        ]

        left, right = refine_window_with_derivative(
            amplitude,
            peak_center,
            gradient
        )

        if right <= left:
            left, right = left_c, right_c

        left = max(left, peak_center - 30)
        right = min(right, peak_center + 30)

        x_win = frequency[left:right]
        y_win = amplitude[left:right]

        if len(x_win) < 5:
            continue

        # intensity profile

        y_raw = y_win - np.min(y_win)

        # keine negativen Werte
        y_raw = np.maximum(y_raw, 0)

        #shape normalised profile
        y_shape = y_raw.copy()

        if np.max(y_shape) > 0:
            y_shape = y_shape / np.max(y_shape)


        analysis = analyze_peak_shape_raw(
            x_win,
            y_shape
        )


        peak_idx = np.argmax(y_shape)

        peak_pos = x_win[peak_idx]

        peak_height = y_shape[peak_idx]


        area = np.trapezoid(
            y_raw,
            x_win
        )


        results.append({

            "cluster_id": i,

            "group_id": peak_to_group.get(
                peak_center,
                None
            ),

            "cluster_peaks": cluster,

            "window": (
                left,
                right
            ),

            "x": x_win,

            "y": y_shape,


            "y_raw": y_raw,


            "peak_position": peak_pos,

            "peak_height": peak_height,

            "area": area,

            "analysis": analysis

        })

    return results

In [ ]:
raw_peak_results = extract_peak_segments_raw(
    frequency,
    amplitude,
    clusters_original,
    peak_to_group
)

Normalization

In [ ]:
def normalize_peak_shape(x, y, mode="l2"):
    # no negative y- Values
    # area of the peak -> creating a probability form
    # l2 = Energy normalization -> comparing peaks independent from the strength of the signal
    # making peaks more comparable
    x = np.asarray(x)
    y = np.maximum(np.asarray(y), 0)

    if len(y) < 3:
        return y

    if mode == "l1":
        area = np.trapezoid(y, x)
        if area > 0:
            y = y / area

    elif mode == "l2":
        norm = np.linalg.norm(y)
        if norm > 0:
            y = y / norm

    return y  

In [ ]:
def build_shape_spectrum_from_clusters(
    results,
    frequency,
    f_min=None,
    f_max=None,
    mode="l2"
):

    spectrum = np.zeros_like(frequency)

    for r in results:

        x = r["x"]
        y = r["y"]

        y = np.maximum(y, 0)
        y = normalize_peak_shape(x, y, mode=mode)

        left = np.searchsorted(frequency, x[0])
        right = left + len(y)

        if right <= len(spectrum):
            spectrum[left:right] += y

    return spectrum

In [ ]:
import numpy as np


def build_raw_spectrum_from_clusters(
    results,
    frequency,
    f_min=None,
    f_max=None
):

    spectrum = np.zeros_like(frequency)

    for r in results:

        x = r["x"]

        # echte Intensität verwenden
        y = r["y_raw"]

        y = np.maximum(y, 0)

        left = np.searchsorted(
            frequency,
            x[0]
        )

        right = left + len(y)

        if right <= len(spectrum):
            spectrum[left:right] += y

    return spectrum

In [ ]:
# Shape Spectrum
spec_orig = build_shape_spectrum_from_clusters(
    raw_peak_results,
    frequency
)
# not normalised spectrum
spec_raw = build_raw_spectrum_from_clusters(
    raw_peak_results,
    frequency
)

In [ ]:
# Debug

print("raw peaks max:",
      max(np.max(r["y"]) for r in raw_peak_results))

print("spec_raw max:",
      np.max(spec_raw))

print("spec_orig max:",
      np.max(spec_orig))

Crosscorrelation and template matching

In [ ]:
import numpy as np
# creating a copy of every cluster 
# every peak has the same length
def build_cluster_template_shape(
    cluster_id,
    raw_peak_results,
    target_len=200
):

    parts = []

    for r in raw_peak_results:

        if r["cluster_id"] != cluster_id:
            continue

        y = np.maximum(r["y"], 0)

        if len(y) < 5:
            continue

        y = y / (np.linalg.norm(y) + 1e-12)

        y = np.interp(
            np.linspace(0, 1, target_len),
            np.linspace(0, 1, len(y)),
            y
        )

        parts.append(y)

    if len(parts) == 0:
        return None

    template = np.mean(parts, axis=0)

    template /= np.linalg.norm(template) + 1e-12

    return template

In [ ]:
# building template for every cluster
def compute_cluster_templates(raw_peak_results):

    cluster_ids = sorted(
        set(r["cluster_id"] for r in raw_peak_results)
    )

    templates = {}

    for cid in cluster_ids:

        template = build_cluster_template_shape(
            cid,
            raw_peak_results
        )

        if template is not None:
            templates[cid] = template

    return templates

In [ ]:
# similarity between signal and template 
def normalized_cross_correlation(signal, template):

    signal = np.asarray(signal)
    template = np.asarray(template)

    signal = signal - np.mean(signal)
    template = template - np.mean(template)
    # cosine similarity matrix
    signal /= np.linalg.norm(signal) + 1e-12
    template /= np.linalg.norm(template) + 1e-12

    return np.correlate(
        signal,
        template,
        mode="full"
    )

In [ ]:
# builds x- axis for correlation
def lag_axis(n, df):

    return np.arange(
        -n + 1,
        n
    ) * df

In [ ]:
from scipy.ndimage import gaussian_filter1d
# looks for periods in the correlation signal
def extract_repetitions_robust(
    corr,
    freq_axis,
    f_min=params["freq_min"],
    f_max=params["freq_max"],
    top_k=5,
    smooth_sigma=2
):

    corr_smooth = gaussian_filter1d(
        corr,
        sigma=smooth_sigma
    )

    corr_smooth /= (
        np.max(np.abs(corr_smooth))
        + 1e-12
    )

    idx = np.argsort(
        corr_smooth
    )[-top_k:]

    positions = np.sort(
        freq_axis[idx]
    )

    positions = positions[
        (positions >= f_min)
        &
        (positions <= f_max)
    ]

    diffs = (
        np.diff(positions)
        if len(positions) > 1
        else []
    )

    return (
        positions,
        diffs,
        corr_smooth
    )

In [ ]:
import numpy as np


def get_cluster_properties(
    raw_peak_results,
    frequency,
    activity=None,
    resonance_mask=None
):

    properties = {}


    for r in raw_peak_results:

        cid = r["cluster_id"]

        pos = r["peak_position"]

        amp = np.max(
            r["y"]
        )


        if activity is not None:

            idx = np.argmin(
                np.abs(frequency-pos)
            )

            peak_activity = activity[idx]

        else:

            peak_activity = 0



        if resonance_mask is not None:

            idx = np.argmin(
                np.abs(frequency-pos)
            )

            in_resonance = bool(
                resonance_mask[idx]
            )

        else:

            in_resonance = False



        if cid not in properties:

            properties[cid] = {
                "positions": [],
                "amplitudes": [],
                "activities": [],
                "resonance": []
            }



        properties[cid]["positions"].append(pos)

        properties[cid]["amplitudes"].append(amp)

        properties[cid]["activities"].append(
            peak_activity
        )

        properties[cid]["resonance"].append(
            in_resonance
        )



    for cid in properties:

        properties[cid]["position"] = np.mean(
            properties[cid]["positions"]
        )

        properties[cid]["amplitude"] = np.mean(
            properties[cid]["amplitudes"]
        )

        properties[cid]["activity"] = np.mean(
            properties[cid]["activities"]
        )

        properties[cid]["in_resonance"] = any(
            properties[cid]["resonance"]
        )


    return properties

In [ ]:
# cosine like similarity between templates 
def compute_similarity_matrix(
    templates
):

    cluster_ids = list(
        templates.keys()
    )

    n = len(cluster_ids)

    sim = np.zeros((n, n))

    for i in range(n):

        for j in range(n):

            t1 = templates[
                cluster_ids[i]
            ]

            t2 = templates[
                cluster_ids[j]
            ]

            sim[i, j] = np.dot(
                t1,
                t2
            )

    return (
        cluster_ids,
        sim
    )

If measured with NTCAP, remove shape if it repeaty way too many times 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Build normalized mean shape for one cluster

def build_cluster_mean_shape(
    cluster_id,
    raw_peak_results,
    target_len=200
):

    parts = []

    for r in raw_peak_results:

        if r["cluster_id"] != cluster_id:
            continue

        y = np.maximum(
            np.asarray(r["y"]),
            0
        )

        if len(y) < 5:
            continue


        norm = np.linalg.norm(y)

        if norm == 0:
            continue

        y = y / norm

        y = np.interp(
            np.linspace(0, 1, target_len),
            np.linspace(0, 1, len(y)),
            y
        )

        parts.append(y)


    if len(parts) == 0:
        return None

    mean_shape = np.mean(
        parts,
        axis=0
    )

    # erneut L2-normalisieren

    mean_shape /= (
        np.linalg.norm(mean_shape)
        + 1e-12
    )

    return mean_shape

# Find groups of clusters with similar shapes

def group_similar_shapes(
    cluster_ids,
    sim_matrix,
    similarity_threshold=0.9
):

    cluster_ids = list(cluster_ids)

    n = len(cluster_ids)

    visited = set()

    shape_groups = []

    for i in range(n):

        cid = cluster_ids[i]

        if cid in visited:
            continue


        group = [cid]

        visited.add(cid)

        for j in range(n):

            if i == j:
                continue

            other = cluster_ids[j]

            if other in visited:
                continue


            similarity = sim_matrix[i, j]


            if similarity >= similarity_threshold:

                group.append(other)

                visited.add(other)


        shape_groups.append(group)


    return shape_groups

# Build averaged shape of several clusters

def build_group_mean_shape(
    group,
    shapes
):

    valid_shapes = [
        shapes[cid]
        for cid in group
        if cid in shapes
    ]

    if len(valid_shapes) == 0:
        return None


    mean_shape = np.mean(
        valid_shapes,
        axis=0
    )



    mean_shape /= (
        np.linalg.norm(mean_shape)
        + 1e-12
    )


    return mean_shape

# optional: Plot 

def plot_cluster_shapes(
    raw_peak_results,
    cluster_positions,
    sim_matrix,
    cluster_ids,
    similarity_threshold=0.9,
    figsize=(12, 8)
):

    shapes = {}

    counts = {}


    for cid in cluster_ids:

        shape = build_cluster_mean_shape(
            cid,
            raw_peak_results,
            target_len=200
        )


        if shape is None:
            continue


        shapes[cid] = shape

        counts[cid] = sum(
            1
            for r in raw_peak_results
            if r["cluster_id"] == cid
        )


    valid_cluster_ids = [
        cid
        for cid in cluster_ids
        if cid in shapes
    ]


    shape_groups = group_similar_shapes(
        valid_cluster_ids,
        sim_matrix,
        similarity_threshold=similarity_threshold
    )


    print("=" * 70)
    print("SHAPE GROUPS")
    print("=" * 70)

    for i, group in enumerate(shape_groups):

        total_peaks = sum(
            counts[cid]
            for cid in group
        )

        print(
            f"Shape {i+1}: "
            f"Clusters = {group} | "
            f"N clusters = {len(group)} | "
            f"N peaks = {total_peaks}"
        )


    print()


    fig, ax = plt.subplots(
        figsize=figsize
    )


    x = np.linspace(
        0,
        1,
        200
    )

    cmap = plt.cm.tab20


    for group_index, group in enumerate(shape_groups):

        group_shape = build_group_mean_shape(
            group,
            shapes
        )


        if group_shape is None:
            continue


        color = cmap(
            group_index % 20
        )


        positions = [
            cluster_positions[cid] / 1e6
            for cid in group
            if cid in cluster_positions
        ]


        total_peaks = sum(
            counts[cid]
            for cid in group
        )


        cluster_text = ", ".join(
            str(cid)
            for cid in group
        )


        label = (
            f"Shape {group_index+1}: "
            #f"C[{cluster_text}] "
            f"(N={total_peaks})"
        )

        # mean shape

        ax.plot(
            x,
            group_shape,
            color=color,
            linewidth=2.5,
            alpha=0.9,
            label=label
        )

        # every peak shown in the background 

        for cid in group:

            ax.plot(
                x,
                shapes[cid],
                color=color,
                alpha=0.12,
                linewidth=0.8
            )



    ax.set_xlabel(
        "Normalized peak position"
    )

    ax.set_ylabel(
        "L2-normalized amplitude"
    )

    ax.set_title(
        "Mean normalized peak shapes grouped by similarity"
    )


    ax.grid(
        alpha=0.2
    )


    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8
    )


    plt.tight_layout()

    plt.show()

    return shape_groups

In [ ]:
templates = compute_cluster_templates(
    raw_peak_results
)


cluster_ids, sim_matrix = compute_similarity_matrix(
    templates
)

cluster_properties = get_cluster_properties(
    raw_peak_results,
    frequency,
    activity=activity,
    resonance_mask=resonance_mask
)

cluster_positions = {
    cid:data["position"]
    for cid,data in cluster_properties.items()
}

shape_groups = plot_cluster_shapes(
    raw_peak_results,
    cluster_positions,
    sim_matrix,
    cluster_ids,
    similarity_threshold=0.99
)

In [ ]:
import numpy as np


def remove_overrepresented_shape_clusters(
    clusters,
    shape_groups,
    expected_repeats=5,
    max_factor=1.8
):

    print("=" * 70)
    print("REMOVE OVERREPRESENTED SHAPE CLUSTERS")
    print("=" * 70)

    max_allowed = expected_repeats * max_factor

    print(
        f"Expected repeats : {expected_repeats}"
    )

    print(
        f"Maximum allowed  : {max_allowed:.1f}"
    )

    print()


    removed_cluster_ids = set()


    for i, shape_group in enumerate(shape_groups):

        print(
            f"Shape {i+1}: "
            f"{len(shape_group)} clusters"
        )

        print(
            f"Clusters: {shape_group}"
        )


        if len(shape_group) > max_allowed:

            print(
                "  -> REMOVE overrepresented shape"
            )

            removed_cluster_ids.update(
                shape_group
            )

        else:

            print(
                "  -> KEEP"
            )

        print()

    all_cluster_ids = set(
        range(len(clusters))
    )

    remaining_cluster_ids = (
        all_cluster_ids
        - removed_cluster_ids
    )


    remaining_cluster_ids = sorted(
        remaining_cluster_ids
    )


    # new clusters with original cluster ids

    clusters_new = [
        clusters[cid]
        for cid in remaining_cluster_ids
    ]


    print("=" * 70)

    print(
        f"Original clusters: "
        f"{len(clusters)}"
    )

    print(
        f"Removed clusters: "
        f"{len(removed_cluster_ids)}"
    )

    print(
        f"Remaining clusters: "
        f"{len(remaining_cluster_ids)}"
    )

    print()

    print(
        "Removed original IDs:"
    )

    print(
        sorted(removed_cluster_ids)
    )

    print()

    print(
        "Remaining original IDs:"
    )

    print(
        remaining_cluster_ids
    )

    print("=" * 70)


    return (
        clusters_new,
        sorted(removed_cluster_ids),
        remaining_cluster_ids
    )


In [ ]:
clusters_new, removed_shape_clusters, remaining_cluster_ids = \
    remove_overrepresented_shape_clusters(
        clusters,
        shape_groups,
        expected_repeats=5,
        max_factor=1.8
    )

In [ ]:
# Filter raw_peak_results according to clusters_new

raw_peak_results_new = [
    r
    for r in raw_peak_results
    if r["cluster_id"] in remaining_cluster_ids
]


print("=" * 70)
print("FILTERED RAW PEAK RESULTS")
print("=" * 70)

print(
    "Original raw peaks:",
    len(raw_peak_results)
)

print(
    "Remaining raw peaks:",
    len(raw_peak_results_new)
)

print(
    "Remaining cluster IDs:",
    sorted(remaining_cluster_ids)
)

In [ ]:
spec_new = build_shape_spectrum_from_clusters(
    raw_peak_results_new,
    frequency
)

Grouping with corrected clusters 

In [ ]:
# Grouping by frequency 
def group_clusters_strict_periodic(
    sim_matrix,
    cluster_ids,
    cluster_positions,
    cluster_amplitudes,
    cluster_activity,
    freq_min=params["freq_min"],
    freq_max=params["freq_max"],
    sim_threshold=0.6,
    min_repeats=params["min_repeats"],
    max_repeats=params["max_repeats"],
    harmonic_tolerance=2e4,
    max_spacing_error=2e4,
    max_harmonic_deviation=0.02   # possible deviation from integer harmonic
):

    candidate_groups = []


    for i, base in enumerate(cluster_ids):

        base_pos = cluster_positions[base]

        possible_frev = []


        for j, other in enumerate(cluster_ids):

            if other == base:
                continue


            if sim_matrix[i,j] < sim_threshold:
                continue


            delta = abs(
                cluster_positions[other]
                -
                base_pos
            )


            if freq_min <= delta <= freq_max:

                possible_frev.append(delta)



        if len(possible_frev) == 0:
            continue



        for frev0 in possible_frev:


            group = []
            harmonics = []


            for h in range(max_repeats+1):

                target = (
                    base_pos
                    +
                    h*frev0
                )


                best = None
                best_score = -np.inf


                for cid in cluster_ids:

                    pos = cluster_positions[cid]


                    error = abs(
                        pos-target
                    )


                    if error > harmonic_tolerance:
                        continue



                    idx1 = cluster_ids.index(base)
                    idx2 = cluster_ids.index(cid)


                    shape = sim_matrix[
                        idx1,
                        idx2
                    ]


                    activity = cluster_activity.get(
                        cid,
                        0
                    )


                    local_score = (

                        0.55*shape

                        +

                        0.25*activity

                        +

                        0.20*np.exp(
                            -0.5*
                            (error/30000)**2
                        )

                    )


                    if local_score > best_score:

                        best_score = local_score
                        best = cid



                if best is not None:

                    if best not in group:

                        group.append(best)
                        harmonics.append(h)



            if len(group) < min_repeats:
                continue



            if len(np.unique(harmonics)) != len(harmonics):
                continue



            # linear fit

            x = np.asarray(
                harmonics
            )


            y = np.asarray([
                cluster_positions[c]
                for c in group
            ])


            fit = np.polyfit(
                x,
                y,
                1
            )


            f_rev = fit[0]
            f0 = fit[1]


            fitted = (
                f0
                +
                x*f_rev
            )


            residuals = (
                y-fitted
            )


            fit_error = np.mean(
                np.abs(residuals)
            )



            if fit_error > max_spacing_error:
                continue



            # check harmonics 

            harmonic_values = (
                (y) / f_rev
            )


            harmonic_deviation = np.abs(
                harmonic_values
                -
                np.round(harmonic_values)
            )


            if np.any(
                harmonic_deviation > max_harmonic_deviation
            ):
                continue



            candidate_groups.append({

                "clusters":group,

                "harmonics":np.asarray(
                    harmonics
                ),

                "harmonic_spacing":f_rev,

                "f0_fit":f0,

                "fit_error":fit_error,

                "harmonic_values":harmonic_values,

                "harmonic_deviation":harmonic_deviation

            })



    # remove double candidates 

    unique = []

    seen = set()


    for g in candidate_groups:

        key = tuple(
            sorted(
                g["clusters"]
            )
        )


        if key in seen:
            continue


        seen.add(key)
        unique.append(g)



    unique = sorted(
        unique,
        key=lambda g: (
            -len(g["clusters"]),
            np.mean(g["harmonic_deviation"]),
            g["fit_error"]
        )
    )



    final = []

    occupied = set()


    for g in unique:

        overlap = len(
            set(g["clusters"])
            &
            occupied
        )


        if overlap > 0:

            if overlap / len(g["clusters"]) > 0.8:
                continue



        final.append(g)

        occupied.update(
            g["clusters"]
        )
    print("\nNumber of candidate groups before overlap filtering:")
    print(len(unique))

    for i,g in enumerate(unique[:20]):

        print(
            i,
            g["clusters"],
            np.round(
                g["harmonic_values"],
                3
            ),
            "f_rev=",
            g["harmonic_spacing"]/1e6
        )

    return final

In [ ]:
# conditions combined 
def score_periodic_groups(
    groups,
    cluster_amplitudes,
    cluster_activity,
    cluster_ids,
    sim_matrix
):

    scored=[]


    max_activity = (
        max(cluster_activity.values())
        +1e-12
    )


    for g in groups:

        clusters = g["clusters"]


        fit_error = g["fit_error"]


        freq_score = np.exp(
            -0.5*(fit_error/15000)**2
        )



        sim_values=[]


        for i in range(len(clusters)):

            for j in range(i+1,len(clusters)):

                idx1=cluster_ids.index(clusters[i])
                idx2=cluster_ids.index(clusters[j])

                sim_values.append(
                    sim_matrix[idx1,idx2]
                )


        shape_score=np.mean(sim_values)



        amps=np.array([
            cluster_amplitudes[c]
            for c in clusters
        ])


        median_amp=np.median(amps)

        amp_dev=np.mean(
            np.abs(
                amps-median_amp
            )
            /
            (median_amp+1e-12)
        )


        amp_score=np.exp(-amp_dev)



        activity_score=np.mean([
            cluster_activity[c]/max_activity
            for c in clusters
        ])



        size_score=min(
            len(clusters)/5,
            1.0
        )
        order_score = g.get(
            "order_score",
            0
        )


        total=(

            0.30*freq_score
            +
            0.15*shape_score
            +
            0.10*amp_score
            +
            0.05*activity_score
            +
            0.05*size_score
            +
            0.35*order_score

        )

        g["order_score"]=order_score
        g["score"]=total
        g["freq_score"]=freq_score
        g["shape_score"]=shape_score
        g["amp_score"]=amp_score
        g["activity_score"]=activity_score
        g["size_score"]=size_score


        scored.append(g)



    return sorted(
        scored,
        key=lambda x:x["score"],
        reverse=True
    )

In [ ]:

# Build templates from remaining peaks

templates = compute_cluster_templates(
    raw_peak_results_new
)


cluster_ids, sim_matrix = compute_similarity_matrix(
    templates
)


cluster_properties = get_cluster_properties(
    raw_peak_results_new,
    frequency,
    activity=activity,
    resonance_mask=resonance_mask
)

# Cluster properties

cluster_positions = {
    cid: data["position"]
    for cid, data in cluster_properties.items()
}


cluster_amplitudes = {
    cid: data["amplitude"]
    for cid, data in cluster_properties.items()
}


cluster_activity = {
    cid: data["activity"]
    for cid, data in cluster_properties.items()
}


valid_cluster_ids = [
    cid
    for cid in cluster_ids
    if cid in cluster_positions
    and cid in cluster_amplitudes
    and cid in cluster_activity
]


print("=" * 70)
print("GROUPING AFTER SHAPE REMOVAL")
print("=" * 70)

print(
    "Remaining clusters:",
    valid_cluster_ids
)

# PERIODIC GROUPING

groups = group_clusters_strict_periodic(
    sim_matrix,
    valid_cluster_ids,
    cluster_positions,
    cluster_amplitudes,
    cluster_activity,
    freq_min=params["freq_min"],
    freq_max=params["freq_max"],
    sim_threshold=0.6,
    min_repeats=params["min_repeats"],
    max_repeats=params["max_repeats"],
    harmonic_tolerance=5e4,
    max_spacing_error=5e4,
    max_harmonic_deviation=0.05
)

# SCORE GROUPS

groups = score_periodic_groups(
    groups,
    cluster_amplitudes,
    cluster_activity,
    valid_cluster_ids,
    sim_matrix
)

# PRINT RESULTS

print("\nDetected periodic groups\n")


for i, g in enumerate(groups):

    print(f"Group {i+1}")

    print(
        f"clusters = {g['clusters']}"
    )

    print(
        f"harmonics = {g['harmonics']}"
    )

    print(
        f"f_rev = {g['harmonic_spacing']/1e6:.6f} MHz"
    )

    print(
        f"fit error = {g['fit_error']/1e3:.2f} kHz"
    )

    print(
        f"score = {g['score']:.3f}"
    )

    print()

In [ ]:
for i, g in enumerate(groups):

    print(f"\n{'='*60}")
    print(f"Group {i+1}")
    print(f"{'='*60}")

    print(
        f"Recovered f_rev : {g['harmonic_spacing']/1e6:.6f} MHz"
    )

    print(
        f"Fit error        : {g['fit_error']/1e3:.2f} kHz"
    )

    print(
        f"Harmonics        : {g['harmonics']}"
    )

    print("\nClusters:")

    for cid, h in zip(g["clusters"], g["harmonics"]):

        print(
            f"  H={h:2d} | "
            f"Cluster {cid:2d} | "
            f"{cluster_positions[cid]/1e6:.6f} MHz | "
            f"Amp = {cluster_amplitudes[cid]:.3f}"
        )

In [ ]:
import copy
import itertools
import numpy as np


def select_final_groups(
    groups,
    min_clusters=2
):
    """
    Selects the combination of groups resulting in the maximum number
    of final groups.

    Rules
    -----
    1. Clusters must not be used more than once.

    2. In the event of a conflict, a cluster may be removed
       from a group.

    3. After removal, a group must contain at least `min_clusters`
       clusters.

    4. Primary objective:
           maximum number of groups

    5. Secondary objective:
           maximum number of clusters

    6. Tertiary objective:
           minimal fit error
    """

    groups = copy.deepcopy(groups)

    if len(groups) == 0:
        return []

    best_solution = None
    best_key = None


    for r in range(1, len(groups) + 1):

        for combination in itertools.combinations(
            groups,
            r
        ):


            combination = sorted(
                combination,
                key=lambda g: len(g["clusters"]),
                reverse=True
            )


            used_clusters = set()

            current_groups = []

            valid = True


            for group in combination:

                clusters = list(
                    group["clusters"]
                )

                harmonics = list(
                    group["harmonics"]
                )


                new_clusters = []
                new_harmonics = []

                for cid, h in zip(
                    clusters,
                    harmonics
                ):

                    if cid in used_clusters:
                        continue

                    new_clusters.append(cid)
                    new_harmonics.append(h)



                if len(new_clusters) < min_clusters:

                    valid = False
                    break


                new_group = copy.deepcopy(group)

                new_group["clusters"] = new_clusters

                new_group["harmonics"] = np.asarray(
                    new_harmonics
                )


                current_groups.append(
                    new_group
                )

                used_clusters.update(
                    new_clusters
                )


            if not valid:
                continue


            number_of_groups = len(
                current_groups
            )


            number_of_clusters = sum(
                len(g["clusters"])
                for g in current_groups
            )


            total_fit_error = sum(
                g.get("fit_error", 0)
                for g in current_groups
            )

            key = (
                number_of_groups,
                number_of_clusters,
                -total_fit_error
            )


            if (
                best_key is None
                or key > best_key
            ):

                best_key = key

                best_solution = current_groups

    if best_solution is None:

        return []
    
    print("=" * 70)
    print("FINAL GROUP SELECTION")
    print("=" * 70)

    print(
        f"Final number of groups: "
        f"{len(best_solution)}"
    )

    print(
        f"Total clusters: "
        f"{sum(len(g['clusters']) for g in best_solution)}"
    )


    for i, g in enumerate(best_solution):

        print(
            f"\nGroup {i+1}"
        )

        print(
            "clusters:",
            g["clusters"]
        )

        print(
            "harmonics:",
            g["harmonics"]
        )

        print(
            f"f_rev = "
            f"{g['harmonic_spacing']/1e6:.6f} MHz"
        )

        print(
            f"fit error = "
            f"{g['fit_error']/1e3:.2f} kHz"
        )


    return best_solution

In [ ]:
groups_final = select_final_groups(
    groups
)

Analysis of the Groups (Positions etc.)

In [ ]:
import numpy as np


def analyze_group_periodicity(
    groups,
    cluster_positions
):

    results = {}


    for gid, g in enumerate(groups):


        clusters = list(
            g["clusters"]
        )


        if len(clusters) < 2:
            continue



        # --------------------------------------------------
        # Frequenzen holen
        # --------------------------------------------------

        frequencies = np.array(
            [
                cluster_positions[c]
                for c in clusters
                if c in cluster_positions
            ]
        )


        clusters = [
            c for c in clusters
            if c in cluster_positions
        ]


        if len(frequencies) < 2:
            continue



        # --------------------------------------------------
        # f_rev aus vorheriger Gruppe
        # --------------------------------------------------

        f_rev_guess = (
            g["harmonic_spacing"]
        )


        if not (
            1.8e6 < f_rev_guess < 2.1e6
        ):
            continue



        # --------------------------------------------------
        # Harmoniken neu bestimmen
        # NICHT übernehmen!
        # --------------------------------------------------

        absolute = (
            frequencies / f_rev_guess
        )


        harmonics = (
            absolute
        )



        # --------------------------------------------------
        # nach Harmoniken sortieren
        # --------------------------------------------------

        order = np.argsort(
            harmonics
        )


        harmonics = harmonics[order]

        frequencies = frequencies[order]

        clusters = (
            np.array(clusters)[order]
            .tolist()
        )



        # --------------------------------------------------
        # Linearer Fit
        # f = f0 + h*f_rev
        # --------------------------------------------------

        fit = np.polyfit(
            harmonics,
            frequencies,
            1
        )


        recovered_frev = fit[0]


        reconstructed = np.polyval(
            fit,
            harmonics
        )


        residuals = (
            frequencies -
            reconstructed
        )


        fit_error = np.mean(
            np.abs(residuals)
        )


        ss_res = np.sum(
            residuals**2
        )


        ss_tot = np.sum(
            (
                frequencies -
                np.mean(frequencies)
            )**2
        )


        r2 = (
            1 -
            ss_res /
            (ss_tot + 1e-12)
        )



        # --------------------------------------------------
        # Ausgabe
        # --------------------------------------------------

        print("\n" + "="*60)
        print(f"Group {gid+1}")
        print("="*60)


        print(
            "Clusters:"
        )

        print(
            clusters
        )


        print(
            "\nPositions"
        )


        for f,h in zip(
            frequencies,
            harmonics
        ):

            print(
                f"{f/1e6:.6f} MHz "
                f" -> harmonic {h}"
            )



        print(
            "\nSpacing"
        )


        spacings = np.diff(
            frequencies
        )


        for d in spacings:

            print(
                f"{d/1e6:.6f} MHz"
            )



        print(
            "\nStatistics"
        )


        print(
            f"Mean spacing : "
            f"{np.mean(spacings)/1e6:.6f} MHz"
        )


        print(
            f"Std spacing  : "
            f"{np.std(spacings)/1e3:.3f} kHz"
        )


        print(
            f"Recovered f_rev : "
            f"{recovered_frev/1e6:.6f} MHz"
        )


        print(
            f"Fit error : "
            f"{fit_error/1e3:.3f} kHz"
        )


        print(
            f"R² : {r2:.8f}"
        )


        print(
            "\nStored:"
        )


        print(
            f"Stored f_rev : "
            f"{g['harmonic_spacing']/1e6:.6f} MHz"
        )



        # --------------------------------------------------
        # speichern
        # --------------------------------------------------

        results[gid] = {

            "clusters":
                clusters,

            "frequencies":
                frequencies,

            "harmonics":
                harmonics,

            "f_rev":
                recovered_frev,

            "fit_error":
                fit_error,

            "r2":
                r2
        }


    return results

In [ ]:
periodicity_results = analyze_group_periodicity(
    groups_final,
    cluster_positions
)

Mapping back to the first harmonic

In [ ]:
import numpy as np
import pandas as pd
# get peaks per cluster and sort them (position wise) 
# only using peaks who can be found in every cluster of the group
# corresponding peaks through the position (first peak to first peak etc)
# revolution frequency for harmonic and revolution frequency = fundamental frequency
def build_peak_families_robust(
    groups,
    raw_peak_results
):

    rows = []

    family_id = 0


    for gid, group in enumerate(groups):


        cluster_ids = group["clusters"]


        cluster_peaks = []

        for cid in cluster_ids:


            peaks = [
                r for r in raw_peak_results
                if r["cluster_id"] == cid
            ]


            peaks = sorted(
                peaks,
                key=lambda x:x["peak_position"]
            )


            if len(peaks):

                cluster_peaks.append(
                    peaks
                )



        if len(cluster_peaks) < 2:
            continue


        min_len = min(
            len(p)
            for p in cluster_peaks
        )

        for peak_index in range(min_len):


            members = [
                peaks[peak_index]
                for peaks in cluster_peaks
            ]


            freqs = np.array([
                m["peak_position"]
                for m in members
            ])


            if len(freqs) < 3:
                continue


            diffs = np.diff(
                np.sort(freqs)
            )


            fundamental = np.median(
                diffs
            )


            if fundamental <= 0:
                continue


            harmonics = (
                freqs / fundamental
            )



            for m,h in zip(
                members,
                harmonics
            ):


                f_peak = m["peak_position"]


                rows.append({

                    "family_id": family_id,

                    "group_id": gid,

                    "cluster_id": m["cluster_id"],

                    "peak_index": peak_index,

                    "f_peak": f_peak,


                    # NEU berechnet
                    "harmonic": h,


                    "fundamental": fundamental,


                    "folded_frequency":
                        f_peak / h
                        if h != 0
                        else np.nan,


                    "window": m["window"]

                })


            family_id += 1


    return pd.DataFrame(rows)

In [ ]:
peak_families = build_peak_families_robust(
    groups_final,
    raw_peak_results
)

In [ ]:
display(
    peak_families.sort_values(
        [
            "group_id",
            "family_id",
            "f_peak"
        ]
    )
    [
        [
            "family_id",
            "group_id",
            "cluster_id",
            "peak_index",
            "f_peak",
            "harmonic",
            "fundamental",
            "folded_frequency"
        ]
    ]
)

In [ ]:
# since first every peak of every cluster gets mapped back to the first harmonic, 
# the peaks at the fundamental frequency need to be averaged out??? 

import numpy as np
import plotly.graph_objects as go
import pandas as pd
from scipy.signal import find_peaks


def merge_close_peaks(peaks, values, min_distance=5, valley_factor=0.7):

    if len(peaks) == 0:
        return []

    peaks = np.array(sorted(peaks))
    merged = []
    i = 0

    while i < len(peaks):

        group = [peaks[i]]
        j = i + 1

        while j < len(peaks):

            if peaks[j] - peaks[i] > min_distance:
                break

            segment = values[peaks[i]:peaks[j] + 1]
            valley = np.min(segment)

            min_height = min(values[peaks[i]], values[peaks[j]])

            if valley > valley_factor * min_height:
                group.append(peaks[j])
                j += 1
            else:
                break

        best = max(group, key=lambda p: values[p])
        merged.append(best)

        i = j

    return np.array(sorted(set(merged)))


def plot_mean_peak_families_styled(
    peak_families,
    frequency,
    spectrum,
    n_points=400,
    prominence_factor=0.02,
    distance=5,
    valley_factor=0.7
):

    fig = go.Figure()

    colors = ["red","blue","green","orange","purple","cyan","magenta","gold"]
    shown = set()

    baseline = 0.005 * np.max(spectrum)

    all_rows = []

    # ============================================================
    # FAMILY LOOP
    # ============================================================
    for fam_id, fam_df in peak_families.groupby("family_id"):

        curves = []

        # -------------------------
        # fold curves
        # -------------------------
        for _, row in fam_df.iterrows():

            left, right = row["window"]
            h = row["harmonic"]

            if h <= 0 or np.isnan(h):
                continue

            l = left
            while l > 0 and spectrum[l] > baseline:
                l -= 1
            if l > 0:
                l -= 1

            r_idx = right
            while r_idx < len(spectrum)-1 and spectrum[r_idx] > baseline:
                r_idx += 1
            if r_idx < len(spectrum)-1:
                r_idx += 1

            x = frequency[l:r_idx]
            y = np.maximum(spectrum[l:r_idx], 0)

            curves.append((x / h, y))

        if len(curves) == 0:
            continue

        xmin = max(np.min(x) for x,_ in curves)
        xmax = min(np.max(x) for x,_ in curves)

        if xmax <= xmin:
            continue

        x_common = np.linspace(xmin, xmax, n_points)

        Y = []
        for x_folded, y in curves:
            Y.append(np.interp(x_common, x_folded, y))

        y_mean = np.maximum(np.nanmean(Y, axis=0), 0)

        # ============================================================
        # PEAK DETECTION
        # ============================================================
        raw_peaks, _ = find_peaks(
            y_mean,
            prominence=prominence_factor * np.max(y_mean),
            distance=distance
        )

        peaks = merge_close_peaks(
            raw_peaks,
            y_mean,
            min_distance=distance,
            valley_factor=valley_factor
        )

        if len(peaks) == 0:
            peaks = [np.nanargmax(y_mean)]

        # ============================================================
        # TABLE
        # ============================================================
        for p in peaks:
            all_rows.append({
                "family_id": fam_id,
                "peak_position": x_common[p],
                "peak_height": y_mean[p]
            })

        # ============================================================
        # PLOT
        # ============================================================
        color = colors[fam_id % len(colors)]

        x_plot = np.concatenate([[x_common[0]], x_common, [x_common[-1]]])
        y_plot = np.concatenate([[0], np.maximum(y_mean, 0), [0]])

        fig.add_trace(go.Scatter(
            x=x_plot,
            y=y_plot,
            mode="lines",
            line=dict(color=color, width=2),
            name=f"Family {fam_id}" if fam_id not in shown else None,
            showlegend=fam_id not in shown
        ))

        shown.add(fam_id)

        fig.add_trace(go.Scatter(
            x=x_common[peaks],
            y=y_mean[peaks],
            mode="markers",
            marker=dict(size=9, color=color),
            showlegend=False
        ))

    peak_table = pd.DataFrame(all_rows)

    print("\n===== PEAK TABLE =====")
    print(peak_table)

    fig.update_layout(
        title="Mean Peak Families (merged + valley-aware)",
        xaxis_title="Frequency (folded)",
        yaxis_title="Amplitude",
        template="plotly_white"
    )

    fig.show()

    return peak_table

In [ ]:
plot_mean_peak_families_styled(
    peak_families,
    frequency,
    spec_orig,
    n_points=4000,
    distance=10,             
    prominence_factor=0.03 
)

Creating a continous spectrum (normalized)

In [ ]:
import numpy as np
import plotly.graph_objects as go


def build_continuous_spectrum_from_mean_families(
    peak_families,
    frequency,
    spectrum,
    n_points=60000,
    baseline_factor=0.005,
    baseline_floor=1e-12
):

    print("=" * 60)
    print("INPUT DEBUG")
    print("=" * 60)
    print("Spectrum max :", np.max(spectrum))
    print("Spectrum min :", np.min(spectrum))
    print("Spectrum id  :", id(spectrum))

    baseline = baseline_factor * np.max(spectrum)

    print("Baseline     :", baseline)

    family_curves = []

    # ============================================================
    # 1. FAMILY CURVES
    # ============================================================
    for fam_id, fam_df in peak_families.groupby("family_id"):

        curves = []

        print("\n-----------------------------------------")
        print(f"Family {fam_id}")

        for _, row in fam_df.iterrows():

            left, right = row["window"]
            h = row["harmonic"]

            if h <= 0 or np.isnan(h):
                continue

            # baseline extension
            l = left
            while l > 0 and spectrum[l] > baseline:
                l -= 1
            if l > 0:
                l -= 1

            r_idx = right
            while r_idx < len(spectrum)-1 and spectrum[r_idx] > baseline:
                r_idx += 1
            if r_idx < len(spectrum)-1:
                r_idx += 1

            x = frequency[l:r_idx]
            y = np.maximum(spectrum[l:r_idx], 0)

            print(
                f"h={h:6.2f} | "
                f"len={len(y):3d} | "
                f"max={np.max(y):.4f} | "
                f"sum={np.sum(y):.4f}"
            )

            curves.append((x / h, y))

        if len(curves) == 0:
            continue

        all_xmins = np.array([np.min(x) for x, _ in curves])
        all_xmaxs = np.array([np.max(x) for x, _ in curves])

        xmin_raw = np.percentile(all_xmins, 5)
        xmax_raw = np.percentile(all_xmaxs, 95)

        pad = 0.2 * (xmax_raw - xmin_raw)

        xmin = xmin_raw - pad
        xmax = xmax_raw + pad

        if xmax <= xmin:
            continue

        x_common = np.linspace(xmin, xmax, n_points)

        Y = []

        for x_folded, y in curves:
            Y.append(np.interp(x_common, x_folded, y))

        y_mean = np.nanmean(Y, axis=0)
        y_mean = np.maximum(y_mean, baseline_floor)

        print(
            f"Family mean: max={np.max(y_mean):.4f}, "
            f"sum={np.sum(y_mean):.4f}"
        )

        family_curves.append((x_common, y_mean))

    if len(family_curves) == 0:
        return None

    x_min = min(c[0][0] for c in family_curves)
    x_max = max(c[0][-1] for c in family_curves)

    x_global = np.linspace(x_min, x_max, n_points)

    stack = []

    for x_c, y_c in family_curves:

        y_interp = np.interp(
            x_global,
            x_c,
            y_c,
            left=baseline_floor,
            right=baseline_floor
        )

        stack.append(y_interp)

    stack = np.array(stack)

    print("\n====================================================")
    print("STACK")
    print("====================================================")

    for i, y in enumerate(stack):
        print(
            f"Family {i}: "
            f"max={np.max(y):.4f} "
            f"sum={np.sum(y):.4f}"
        )

    spectrum_cont = np.nanmean(stack, axis=0) * len(stack)

    print("\nContinuous before normalization")
    print("max =", np.max(spectrum_cont))
    print("sum =", np.sum(spectrum_cont))

    spectrum_cont /= np.max(spectrum_cont)

    print("\nContinuous after normalization")
    print("max =", np.max(spectrum_cont))
    print("sum =", np.sum(spectrum_cont))

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=x_global,
        y=spectrum_cont,
        mode="lines",
        line=dict(color="black", width=2),
        name="Continuous spectrum"
    ))

    fig.update_layout(
        title="Continuous Spectrum",
        xaxis_title="Frequency",
        yaxis_title="Amplitude",
        template="plotly_white"
    )

    fig.show()

    return x_global, spectrum_cont

In [ ]:
print("\n====================")
print("SPEC_ORIG")
print("====================")

x_cont_orig, spec_cont_orig = build_continuous_spectrum_from_mean_families(
    peak_families,
    frequency,
    spec_orig,
    n_points=600000
)

In [ ]:
x_cont_x, spec_cont_x = build_continuous_spectrum_from_mean_families(
    peak_families,
    frequency,
    spec_orig,
    n_points=60000,
    baseline_floor=1e-12
)

np.savez(
    "continuous_spectrum_x.npz",
    frequency=x_cont_x,
    spectrum=spec_cont_x
)

Continous spectrum not normalized 

In [ ]:
print("\n====================")
print("SPEC_RAW")
print("====================")

x_cont_raw_x, spec_cont_raw_x = build_continuous_spectrum_from_mean_families(
    peak_families,
    frequency,
    spec_raw,
    n_points=600000
)

In [ ]:
x_cont_raw_x, spectrum_cont_raw_x = build_continuous_spectrum_from_mean_families(
    peak_families,
    frequency,
    spec_raw,
    n_points = 600000
)

np.savez(
    "continuous_spectrum_raw_x.npz",
    frequency=x_cont_raw_x,
    spectrum=spectrum_cont_raw_x
)

Positions and frequency differences

In [ ]:
import numpy as np
import pandas as pd
# analysis with fwhm etc. and pairwise frequency differences 
def build_frequency_analysis_table(peak_families, frequency, spectrum):

    rows = []

    for _, entry in peak_families.iterrows():

        gid = entry["group_id"]
        cid = entry["cluster_id"]

        harmonic = entry["harmonic"]

        if harmonic <= 0 or np.isnan(harmonic):
            continue

        left, right = entry["window"]

        x = frequency[left:right] / harmonic
        y = spectrum[left:right]

        mask = ~np.isnan(y)
        if np.sum(mask) < 3:
            continue

        x_valid = x[mask]
        y_valid = y[mask]

        peak_idx = np.nanargmax(y_valid)
        peak_pos_folded = x_valid[peak_idx]

        peak_height = y_valid[peak_idx]
        half_max = peak_height / 2

        left_idx = peak_idx
        while left_idx > 0 and y_valid[left_idx] > half_max:
            left_idx -= 1

        right_idx = peak_idx
        while right_idx < len(y_valid) - 1 and y_valid[right_idx] > half_max:
            right_idx += 1

        if right_idx <= left_idx:
            continue

        fwhm_folded = x_valid[right_idx] - x_valid[left_idx]
        sigma_folded = fwhm_folded / (2 * np.sqrt(2 * np.log(2)))

        peak_pos_physical = peak_pos_folded * harmonic

        rows.append({
            "group_id": gid,
            "cluster_id": cid,

            "harmonic": harmonic,

            # folded domain
            "peak_position_folded": peak_pos_folded,
            "fwhm_folded": fwhm_folded,
            "sigma_folded": sigma_folded,

            # physical domain
            "peak_position_Hz": peak_pos_physical,
            "peak_position_MHz": peak_pos_physical / 1e6,

            "fwhm_Hz": fwhm_folded * harmonic,
            "sigma_Hz": sigma_folded * harmonic
        })

    df = pd.DataFrame(rows)

    if len(df) == 0:
        return df, pd.DataFrame()

    df = df.sort_values("peak_position_Hz").reset_index(drop=True)

    #  PAIRWISE DIFFERENCES

    diff_rows = []

    positions = df["peak_position_Hz"].values
    sigmas = df["sigma_Hz"].values

    gids = df["group_id"].values
    cids = df["cluster_id"].values
    harmonics = df["harmonic"].values

    for i in range(len(df)):
        for j in range(i + 1, len(df)):

            delta_f = positions[j] - positions[i]
            delta_sigma = np.sqrt(sigmas[i]**2 + sigmas[j]**2)

            rel_df = delta_f / positions[i] if positions[i] != 0 else np.nan

            diff_rows.append({
                "group_1": gids[i],
                "group_2": gids[j],

                "cluster_1": cids[i],
                "cluster_2": cids[j],

                "harmonic_1": harmonics[i],
                "harmonic_2": harmonics[j],

                "f1_Hz": positions[i],
                "f2_Hz": positions[j],

                "delta_f_Hz": delta_f,
                "abs_delta_f_Hz": abs(delta_f),

                "sigma_delta_f_Hz": delta_sigma,
                "relative_df_over_f": rel_df
            })

    df_diff = pd.DataFrame(diff_rows)

    df_diff = df_diff.sort_values("abs_delta_f_Hz").reset_index(drop=True)

    # OUTPUT

    print("\n==============================")
    print("PEAK POSITIONS (CONSISTENT)")
    print("==============================")

    display(df[[
        "group_id",
        "cluster_id",
        "harmonic",
        "peak_position_MHz",
        "sigma_Hz"
    ]])

    print("\n==============================")
    print("PAIRWISE DIFFERENCES")
    print("==============================")

    display(df_diff)

    return df, df_diff

In [ ]:
import numpy as np
import pandas as pd
# nearest neightbor analysis 
df = peak_families.copy()

fundamentals = df.sort_values("f_peak").reset_index(drop=True)

freqs = fundamentals["f_peak"].values

print("\n===== PEAK POSITIONS (SORTED) =====")

display(
    fundamentals[
        [
            "family_id",
            "cluster_id",
            "f_peak",
            "harmonic",
            "fundamental"
        ]
    ]
)

#  NEAREST NEIGHBORS

diff_rows = []

for i in range(len(fundamentals) - 1):

    j = i + 1

    f1 = freqs[i]
    f2 = freqs[j]

    delta_f = f2 - f1

    diff_rows.append({
        "family_1": fundamentals["family_id"].iloc[i],
        "family_2": fundamentals["family_id"].iloc[j],

        "cluster_1": fundamentals["cluster_id"].iloc[i],
        "cluster_2": fundamentals["cluster_id"].iloc[j],

        "f1_Hz": f1,
        "f2_Hz": f2,

        "delta_f_Hz": delta_f,
        "relative_df": delta_f / f1 if f1 != 0 else np.nan
    })

df_diff = pd.DataFrame(diff_rows)

df_diff = df_diff.sort_values("delta_f_Hz").reset_index(drop=True)

print("\n===== NEAREST NEIGHBOR SPACING =====")

display(df_diff)